In [ ]:
print("ok")

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    
    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf_file(data='Data/')

In [ ]:
extracted_data

In [ ]:
# split the data in to text chunks

def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks = text_split(extracted_data)
print(f"Number of text chunks: {len(text_chunks)}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


In [ ]:
# download embeddings from huggingface
def download_huggingface_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
embeddings = download_huggingface_embeddings() 

In [ ]:
embeddings

In [ ]:
query_results = embeddings.embed_query("Hello World")
print(f"Query results: {query_results}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY= os.getenv("OPENAI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY1")


In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chat-boat"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

In [ ]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["OPENROUTER_API_KEY1"] = OPENROUTER_API_KEY

In [ ]:
# Embed each  chunk  and upsert the embeddings into our pinecone index

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
# load existing index

from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
docsearch

In [ ]:
retriever = docsearch.as_retriever(search_type = "similarity", search_kwargs={"k": 3})

In [ ]:
retriever_docs = retriever.invoke("What is Acne?")

In [ ]:
retriever_docs

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-5.6-luna",
    api_key=os.getenv("Openai_API_KEY"),
    temperature=0.4,
)

In [ ]:
response = llm.invoke("Hello")
print(response)

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for quetion-answering tasks. "
    "Use the following pieces of retrived context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the answer consise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input":"What is Acne?"})
print(f"Response: {response['answer']}")